In [1]:
import importlib
import pipeline.bronze
importlib.reload(pipeline.bronze)

from pipeline.bronze import fetch_source, SOURCES

response = fetch_source("online_retail_ii")

print("Källa:", SOURCES["online_retail_ii"]["url"])
print("Status code:", response.status_code)
print("Content length (bytes):", len(response.content))

# Test
# expected: 200, ~45.6 MB

Källa: https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip
Status code: 200
Content length (bytes): 45622418


In [2]:
import zipfile
import io

with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
    print(zf.namelist())

['online_retail_II.xlsx']


In [3]:
importlib.reload(pipeline.bronze)
from pipeline.bronze import extract_from_zip

inner_name = "online_retail_II.xlsx"  #from previous cell

xlsx_bytes = extract_from_zip(response.content, inner_name)

print("size:", len(xlsx_bytes))

size: 45622278


In [4]:
# sanity check: en .xlsx-fil är själv ett zip-arkiv, och börjar därför
# alltid med samma fyra byte-signatur (PK\x03\x04)
# had to grab AI help for the byte signature. for asserts
assert len(xlsx_bytes) > 0
assert xlsx_bytes[:4] == b"PK\x03\x04", "Ser inte ut som en giltig xlsx/zip-fil"

In [5]:
import importlib
import pipeline.bronze
importlib.reload(pipeline.bronze)

from pipeline.bronze import fetch_and_extract

xlsx_bytes = fetch_and_extract("online_retail_ii")

In [6]:
assert xlsx_bytes[:4] == b"PK\x03\x04"
print("fetch_and_extract: OK")

fetch_and_extract: OK


Takes about 40sec, downloads 45 MB, unzips it, and parses the xlsx

In [9]:
from pipeline.convert import xlsx_to_dataframe, dataframe_to_parquet_bytes

xlsx_bytes = fetch_and_extract("online_retail_ii")
sheets = xlsx_to_dataframe(xlsx_bytes)

for name, df in sheets.items():
    print(name, df.shape)
    print(list(df.columns))

Year 2009-2010 (525461, 8)
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']
Year 2010-2011 (541910, 8)
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


In [10]:
import pandas as pd

frames = list(sheets.values())
assert all(list(f.columns) == list(frames[0].columns) for f in frames)

df_retail = pd.concat(frames, ignore_index=True)
assert len(df_retail) == sum(len(f) for f in frames)
print(df_retail.shape)

(1067371, 8)


In [11]:
"""parquet_retail = dataframe_to_parquet_bytes(df_retail)
roundtrip = pd.read_parquet(io.BytesIO(parquet_retail))

pd.testing.assert_frame_equal(roundtrip, df_retail)
print(f"xlsx:    {len(xlsx_bytes) / 1e6:.1f} MB")
print(f"parquet: {len(parquet_retail) / 1e6:.1f} MB")"""

ArrowInvalid: ("Could not convert 'C489449' with type str: tried to convert to int64", 'Conversion failed for column Invoice with type object')

### Design choice
the code above is throwing an error from parsing invoice with a C prefix, when invoices are majoritly int64, so it first assigned the column as an int, but then a C prefix got attached, need to clean this up. possibly with a seperation of issues so i grab mixed columns and then extract them into a separate table where its read as str instead of int, and the "master" doc has the parameter set to true null rather than nan

In [12]:
from pipeline.convert import xlsx_bytes_to_parquet
parquet, meta = xlsx_bytes_to_parquet(xlsx_bytes)
out = pd.read_parquet(io.BytesIO(parquet))

# ingen dataförlust
assert meta["rows_in"] == meta["rows_out"] == len(out) == 1_067_371
assert list(out.columns) == meta["columns"]

# inga null blev till texten "nan" på vägen
nulls_before = sum(df.isna().sum().sum() for df in sheets.values())
assert out.isna().sum().sum() == nulls_before

print(meta)

{'sheets': ['Year 2009-2010', 'Year 2010-2011'], 'rows_in': 1067371, 'rows_out': 1067371, 'columns': ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country'], 'cast_to_str': ['Invoice', 'StockCode', 'Description'], 'parquet_bytes': 6058397}


In [13]:
from pipeline.convert import find_mixed_type_columns
print(find_mixed_type_columns(df_retail)["Description"])

odd = df_retail["Description"].dropna()
odd = odd[odd.map(type) != str]
print(odd.head(10).tolist())

{'str': 1062985, 'int': 4}
[21494, 22719, 22467, 20713]


In [15]:
import importlib, time, hashlib, json
import pipeline.bronze
importlib.reload(pipeline.bronze)
from pipeline.bronze import get_raw, BRONZE_DIR

t0 = time.perf_counter()
raw_1 = get_raw("online_retail_ii")      # första gången: hämtar och sparar
t1 = time.perf_counter()
raw_2 = get_raw("online_retail_ii")      # andra gången: läser från disk
t2 = time.perf_counter()

meta = json.loads((BRONZE_DIR / "online_retail_ii.meta.json").read_text())

assert raw_1 == raw_2
assert meta["sha256"] == hashlib.sha256(raw_1).hexdigest()
assert meta["size_bytes"] == len(raw_1)

print(f"första:  {t1 - t0:.1f} s")
print(f"andra:   {t2 - t1:.1f} s")
print(meta)

första:  4.2 s
andra:   0.0 s
{'source': 'online_retail_ii', 'filename': 'online_retail_II.xlsx', 'sha256': 'bcbe73b35f5b7babf197fb0cb983a11f5d9ff929078d4aa53d171b1f2df2e980', 'size_bytes': 45622278, 'ingested_at': '2026-11-25T03:15:10.197199+00:00'}


cached data works, everything appears to be nominal, will need to do extensive testing for later,

now im testing merely to make sure it follows the intended purpose.

But i will need to place more checks and balances later when finetuning.

In [16]:
df_retail.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
